In [17]:
import os
import pandas as pd
import gseapy as gp
import plotly.express as px
import json
import numpy as np
from gseapy import Biomart
import plotly.graph_objs as go
from pathlib import Path
from itables import show

## ORA

In [18]:
contrast = ""
df = pd.read_csv(f"./results/contrasts/{contrast}.csv",sep=';')
df

,gene_id,gene_name,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
0,ENSMUSG00000000001,Gnai3,6614.435806,0.071631,0.061170,1.171024,0.241589,0.376763
1,ENSMUSG00000000028,Cdc45,2677.765983,-0.053403,0.067646,-0.789450,0.429849,0.566522
2,ENSMUSG00000000031,H19,3.545939,0.756689,0.707121,1.070097,0.284575,NaN
3,ENSMUSG00000000037,Scml2,506.453130,-0.430428,0.134006,-3.211999,0.001318,0.010213
4,ENSMUSG00000000056,Narf,2833.805782,-0.379488,0.161120,-2.355315,0.018507,0.061281
...,...,...,...,...,...,...,...,...
22284,ENSMUSG00002076556,Gm56424,16.984368,-0.763153,0.990742,-0.770284,0.441131,0.576667
22285,ENSMUSG00002076601,Scarna13,0.616981,-0.113076,2.031281,-0.055667,0.955607,NaN
22286,ENSMUSG00002076665,Gm54427,0.485802,1.202252,2.602915,0.461887,0.644162,NaN
22287,ENSMUSG00002076675,Gm56334,0.729818,-0.044038,1.805741,-0.024388,0.980543,NaN


In [19]:
def compute_ORA(df,database_name='GO_Biological_Process_2023',organism='Mouse'):
    gene_list = df[df['padj']<0.05]['gene_name'].tolist()
    # if you are only intrested in dataframe that enrichr returned, please set outdir=None
    enr = gp.enrichr(gene_list=gene_list, # or "./tests/data/gene_list.txt",
                    gene_sets=[database_name],
                    organism=organism, # don't forget to set organism to the one you desired! e.g. Yeast
                    outdir=None, # don't write to disk
                    )
    ora_results = enr.results
    ora_results['Overlap_Ratio'] = ora_results['Overlap'].apply(lambda x: int(x.split('/')[0]) / int(x.split('/')[1]))
    ora_results['Count'] = ora_results['Overlap'].apply(lambda x: int(x.split('/')[0]))
    return ora_results

In [20]:
#ora_res = compute_ORA(df)
#ora_res_up = compute_ORA(df[df['log2FoldChange']>0])
#ora_res_down = compute_ORA(df[df['log2FoldChange']<0])

In [21]:
#ora_res.to_excel(f"./results/excel/pathway/ORA/ORA_GO_{contrast}_ALL.xlsx")
#ora_res_up.to_excel(f"./results/excel/pathway/ORA/ORA_GO_{contrast}_UP.xlsx")
#ora_res_down.to_excel(f"./results/excel/pathway/ORA/ORA_GO_{contrast}_DOWN.xlsx")

In [43]:
def full_go_analysis(df,contrast):
    ora_res = compute_ORA(df)
    ora_res.to_csv(f"./results/pathway/ORA/ORA_GO_{contrast}_ALL.csv",sep=';')
    
    ora_res_up = compute_ORA(df[df['log2FoldChange']>0])
    ora_res_up.to_csv(f"./results/pathway/ORA/ORA_GO_{contrast}_UP.csv",sep=';')
    
    ora_res_down = compute_ORA(df[df['log2FoldChange']<0])
    ora_res_down.to_csv(f"./results/pathway/ORA/ORA_GO_{contrast}_DOWN.csv",sep=';')

In [44]:
full_go_analysis(df,contrast)

## Databases

In [24]:
mouse = gp.get_library_name(organism='Mouse')

In [25]:
mouse

['ARCHS4_Cell-lines',
 'ARCHS4_IDG_Coexp',
 'ARCHS4_Kinases_Coexp',
 'ARCHS4_TFs_Coexp',
 'ARCHS4_Tissues',
 'Achilles_fitness_decrease',
 'Achilles_fitness_increase',
 'Aging_Perturbations_from_GEO_down',
 'Aging_Perturbations_from_GEO_up',
 'Allen_Brain_Atlas_10x_scRNA_2021',
 'Allen_Brain_Atlas_down',
 'Allen_Brain_Atlas_up',
 'Azimuth_2023',
 'Azimuth_Cell_Types_2021',
 'BioCarta_2013',
 'BioCarta_2015',
 'BioCarta_2016',
 'BioPlanet_2019',
 'BioPlex_2017',
 'CCLE_Proteomics_2020',
 'CM4AI_U2OS_Protein_Localization_Assemblies',
 'COMPARTMENTS_Curated_2025',
 'COMPARTMENTS_Experimental_2025',
 'CORUM',
 'COVID-19_Related_Gene_Sets',
 'COVID-19_Related_Gene_Sets_2021',
 'Cancer_Cell_Line_Encyclopedia',
 'Carcinogenome',
 'CellMarker_2024',
 'CellMarker_Augmented_2021',
 'ChEA_2013',
 'ChEA_2015',
 'ChEA_2016',
 'ChEA_2022',
 'Chromosome_Location',
 'Chromosome_Location_hg19',
 'ClinVar_2019',
 'ClinVar_2025',
 'DGIdb_Drug_Targets_2024',
 'DSigDB',
 'Data_Acquisition_Method_Most_Popul

In [26]:
#go_bp = gp.get_library(name='GO_Biological_Process_2023', organism='Mouse')

In [27]:
"""with open("../02_DB/Mouse_GO_Biological_Process_2023.json", "w") as outfile: 
    json.dump(go_bp, outfile)"""

'with open("../02_DB/Mouse_GO_Biological_Process_2023.json", "w") as outfile: \n    json.dump(go_bp, outfile)'

In [28]:
HM = gp.get_library(name='MSigDB_Hallmark_2020', organism='Mouse')

In [29]:
with open("../02_DB/Mouse_MSigDB_Hallmark_2020", "w") as outfile: 
    json.dump(HM, outfile)

## GSEA

In [30]:
# Read data
with open('../02_DB/Mouse_GO_Biological_Process_2023.json', 'r') as f:
    mouse_go = json.load(f)

with open('../02_DB/Mouse_KEGG_2019_Mouse.json', 'r') as f:
    mouse_KEGG = json.load(f)
    
with open('../02_DB/Mouse_Reactome_Pathways_2024.json', 'r') as f:
    mouse_reactome = json.load(f)

with open('../02_DB/Mouse_MSigDB_Hallmark_2020', 'r') as f:
    HM = json.load(f)


In [37]:
import numpy as np
import pandas as pd
import gseapy as gp

def compute_GSEA(df: pd.DataFrame, database: str) -> pd.DataFrame:
    """
    Compute preranked GSEA using gseapy with ranking = sign(log2FC) * -log10(pvalue).

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing at least 'gene_name', 'log2FoldChange', and 'pvalue'.
    database : str
        Gene set database path or name (GMT file or Enrichr library).

    Returns
    -------
    gsea_res : pd.DataFrame
        GSEA results with additional columns:
        - 'Tag_%' : fraction of genes in the gene set
        - '-log10Qvalue' : -log10(FDR q-value)
    """

    # Remove genes with missing p-values
    df_cleaned = df.dropna(subset=['pvalue']).copy()

    # Compute rank: sign(log2FC) * -log10(pvalue)
    df_cleaned['rank'] = df_cleaned.apply(
        lambda x: np.sign(x['log2FoldChange']) * -np.log10(x['pvalue']),
        axis=1
    )

    # Prepare prerank input: unique gene names, uppercase
    rnk = (
        df_cleaned[['gene_name', 'rank']]
        .sort_values(by='rank', ascending=False)
        .drop_duplicates(subset=['gene_name'], keep='first')
        .dropna()
    )
    rnk['gene_name'] = rnk['gene_name'].str.upper()

    # Run preranked GSEA
    pre_res = gp.prerank(
        rnk=rnk,
        gene_sets=database,
        min_size=10,
        max_size=500,
        permutation_num=1000,  # reduce for speed in testing
        outdir=None,            # do not write to disk
        seed=6,
        verbose=False,
        threads=12,
    )

    gsea_res = pre_res.res2d.copy()

    # Compute fraction of genes in the gene set (Tag %)
    tag_split = gsea_res['Tag %'].str.split("/", expand=True)
    gsea_res['Tag_%'] = tag_split[0].astype(int) / tag_split[1].astype(int)

    # Handle zero FDR values for -log10 transformation
    non_zero_fdr_min = gsea_res[gsea_res['FDR q-val'] != 0.0]['FDR q-val'].min()
    gsea_res['FDR q-val non zero'] = gsea_res['FDR q-val'].apply(
        lambda x: x if x != 0.0 else non_zero_fdr_min
    )
    gsea_res['-log10Qvalue'] = -np.log10(gsea_res['FDR q-val non zero'])

    return gsea_res


In [41]:
def full_gsea_analysis(df,contrast):
    # Read data
    with open('../02_DB/Mouse_GO_Biological_Process_2023.json', 'r') as f:
        mouse_go = json.load(f)

    with open('../02_DB/Mouse_KEGG_2019_Mouse.json', 'r') as f:
        mouse_KEGG = json.load(f)
    
    with open('../02_DB/Mouse_MSigDB_Hallmark_2020', 'r') as f:
        mouse_HM = json.load(f)
    #GO database
    gsea_res_mouse_go = compute_GSEA(df,mouse_go)
    gsea_res_mouse_go.to_csv(f"./results/pathway/GSEA/GSEA_GO_{contrast}.csv",sep=';')
    
    #KEGG database
    gsea_res_mouse_KEGG = compute_GSEA(df,mouse_KEGG)
    gsea_res_mouse_KEGG.to_csv(f"./results/pathway/GSEA/GSEA_KEGG_{contrast}.csv",sep=';')

    #Wiki database
    gsea_res_HM_reactome = compute_GSEA(df,mouse_HM)
    gsea_res_HM_reactome.to_csv(f"./results/pathway/GSEA/GSEA_HM_{contrast}.csv",sep=';')

In [42]:
full_gsea_analysis(df,contrast)

In [47]:
path = "./results/contrasts/"

for file in os.listdir(path):
    if os.path.isfile(os.path.join(path, file)):
        df = pd.read_csv(f"{path}{file}",sep=';')
        contrast = Path(file).stem
        print("compute ORA for: ",contrast)
        full_go_analysis(df,contrast)
        print("compute GSEA for: ",contrast)
        full_gsea_analysis(df,contrast)
    else:
        print("this is not a file")

compute ORA for:  condition_Control_Old_VS_Control_Young
compute GSEA for:  condition_Control_Old_VS_Control_Young
compute ORA for:  condition_Trained_Old_VS_Control_Old
compute GSEA for:  condition_Trained_Old_VS_Control_Old
compute ORA for:  condition_Trained_Old_VS_Trained_Young
compute GSEA for:  condition_Trained_Old_VS_Trained_Young
compute ORA for:  condition_Trained_Young_VS_Control_Young
compute GSEA for:  condition_Trained_Young_VS_Control_Young
